In [2]:
# %pip install -r "/Workspace/Users/grazid01@heiway.net/libs/requirements.txt" 
# %pip install "/Workspace/Users/grazid01@heiway.net/libs/svoc-0.1.0-py3-none-any.whl" 
# dbutils.library.restartPython()

### Import

In [1]:
import json
from svoc.settings import get_settings
from svoc.utils import read_data_from_csv
from svoc.datapreparation import prepare_data
from svoc.rl import get_matches_with_clusters, prepare_output
from svoc.orchestrator import svoc_knn
import pandas as pd
import numpy as np
from svoc.constants import DISTANCES, FILTERS_AUTO

### Data Preparation

In [2]:
settings = get_settings()
df_input, df_benchmark = read_data_from_csv(settings)

In [4]:
df_input[settings.INPUT_COLUMNS.ID] = [str(num) + "___" + id for num, id in zip(range(0, len(df_input)), df_input[settings.INPUT_COLUMNS.ID])] 

In [5]:
df_benchmark_clean = prepare_data(
    df=df_benchmark, dict_cols=settings.BENCHMARK_COLUMNS_DICT)
df_input_clean = prepare_data(
    df=df_input, dict_cols=settings.INPUT_COLUMNS_DICT)

### KNN

In [7]:
groups = svoc_knn(
        settings=settings, 
        df_input=df_input, 
        df_benchmark=df_benchmark, 
        k=settings.K_NEIGHBOURS,
        save=False
    )

### Matching

In [10]:
all_matches, features, remaining_features = get_matches_with_clusters(
    df_input=df_input_clean.head(100), 
    df_benchmark=df_benchmark_clean, 
    distances=DISTANCES, 
    filters=FILTERS_AUTO,
    block_col=settings.BLOCK_COL,
    groups=groups,
    n_matches=settings.N_BENCHMARK_MATCHES,
    models_path_dict=settings.SUPERVISED_MODELS_PATHS,
    verbose=False
    )

In [12]:
all_matches

,ID_1,ID_2,outlet_name_cosine,address_cosine,postcode,outlet_name_jarowinkler,outlet_name_levenshtein,outlet_name_qgram,outlet_name_clean_cosine,outlet_name_clean_jarowinkler,...,address_in2,address_clean_in2,name_address_in2,address_name_in2,score,ID_filter,match_type,_prob,model,rank
236,1001107,3___7018146,0.129099,0.672194,0,0.490764,0.172414,0.100000,0.129099,0.490764,...,1,1,0,0,0.373403,47.0,auto,NaN,NaN,1
541,1001336,1___93119,0.000000,0.657411,0,0.450000,0.100000,0.000000,0.000000,0.450000,...,0,0,0,0,0.393754,56.0,auto,NaN,NaN,1
542,1001369,34___91512,0.056614,0.032686,0,0.492063,0.095238,0.045455,0.056614,0.492063,...,0,0,0,0,0.207916,56.0,auto,NaN,NaN,1
237,1001375,44___G19214,0.048002,0.468293,0,0.498042,0.137931,0.034483,0.048002,0.500916,...,1,1,0,0,0.308619,47.0,auto,NaN,NaN,1
238,1001601,81___L06037,0.000000,0.804668,1,0.500000,0.166667,0.000000,0.000000,0.500000,...,1,1,0,0,0.419545,47.0,auto,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,97828,58___CROWN INN,0.824958,0.033333,0,0.911111,0.555556,0.600000,0.824958,0.911111,...,0,0,0,0,0.478386,NaN,supervised,0.518355,logreg,1
37,98332,65___7030214,0.806226,0.938953,1,0.917647,0.588235,0.611111,0.806226,0.917647,...,1,1,0,0,0.775182,20.0,auto,NaN,NaN,1
219,99307,58___CROWN INN,0.640513,0.957427,0,0.602694,0.272727,0.545455,0.640513,0.614815,...,1,1,0,0,0.720618,41.0,auto,NaN,NaN,1
447,99313,58___CROWN INN,0.481125,0.731564,0,0.564815,0.444444,0.400000,0.481125,0.564815,...,1,1,0,0,0.521073,47.0,auto,NaN,NaN,1


In [13]:
output = prepare_output(
        matches=all_matches,
        distances=DISTANCES,
        filters=FILTERS_AUTO,
        max_input_matches=settings.N_INPUT_MATCHES
    )

In [14]:
output

,ID_1,ID_2,ID_filter,rank,score,match_type,model,OUTLET_NAME_score,ADDRESS_score,POSTCODE_score,OUTLET_NAME_method,ADDRESS_method,POSTCODE_method,ADDRESS_OUTLET_NAME_score,ADDRESS_OUTLET_NAME_method,OUTLET_NAME_ADDRESS_score,OUTLET_NAME_ADDRESS_method
231,1001107,3___7018146,47.0,1,0.373403,auto,NaN,0.129099,1.000000,0.866667,cosine,wordsmatch,jarowinkler,NaN,NaN,NaN,NaN
281,1001601,81___L06037,47.0,1,0.419545,auto,NaN,0.000000,1.000000,1.000000,cosine,wordsmatch,jarowinkler,NaN,NaN,NaN,NaN
141,1001731,5___7032265,41.0,1,0.452280,auto,NaN,0.664573,0.636887,0.866667,jarowinkler,jarowinkler,jarowinkler,NaN,NaN,NaN,NaN
238,1002080,42___15277,47.0,1,0.534572,auto,NaN,0.000000,1.000000,1.000000,cosine,wordsmatch,jarowinkler,NaN,NaN,NaN,NaN
355,1003055,46___BORDESLEY PARK FARM,56.0,1,0.370362,auto,NaN,0.067420,0.496573,0.000000,cosine,cosine,exact,0.834501,jarowinkler,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
424,97828,58___CROWN INN,NaN,1,0.478386,supervised,logreg,0.824958,0.033333,0.000000,cosine,cosine,exact,NaN,NaN,NaN,NaN
30,98332,65___7030214,20.0,1,0.775182,auto,NaN,1.000000,1.000000,1.000000,wordsmatch,wordsmatch,exact,NaN,NaN,NaN,NaN
137,99307,58___CROWN INN,41.0,1,0.720618,auto,NaN,0.614815,0.848694,0.828571,jarowinkler,jarowinkler,jarowinkler,NaN,NaN,NaN,NaN
246,99313,58___CROWN INN,47.0,1,0.521073,auto,NaN,0.481125,1.000000,0.828571,cosine,wordsmatch,jarowinkler,NaN,NaN,NaN,NaN


## Analisi risultati

In [15]:
df_input[df_input['outletname']=="CROWN INN"]

,bvtsvoc_name,ontoftcode,outletname,outletaddress,outletpostcode,latitude,longitude
58,Small Beer,58___CROWN INN,CROWN INN,12 BRIDGE STREET DOWNHAM MARKET,PE38 9DH,52.602928,0.376166
305,Small Beer,305___CROWN INN,CROWN INN,NaN,PE38 9DH,52.602928,0.376166
480,Small Beer,480___CROWN INN,CROWN INN,12 BRIDGE STREET DOWNHAM MARKET,NaN,NaN,NaN


In [ ]:
output.to_csv("C:\\data\\HUKsvoc\\data\\20260403_output.csv", index=False)

In [ ]:
match_per_input = output.groupby('ID_2')['ID_1'].count().reset_index().sort_values('ID_1', ascending=False).rename(columns={'ID_1': 'count'})
list_idx_input = match_per_input[match_per_input['count'] > 100]['ID_2'].tolist()
match_per_input

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Istogramma con seaborn
plt.figure(figsize=(10, 6))
sns.histplot(data=match_per_input['count'].dropna(), bins='auto', kde=True, color='#2E86AB')
plt.title("Istogramma della colonna 'count' - match_per_input")
plt.xlabel("Valori di count")
plt.ylabel("Frequenza")
plt.grid(axis='y', alpha=0.3)

# Salva il grafico
plt.tight_layout()
plt.show()

In [ ]:
output_full = (output
 .merge(df_benchmark_clean, left_on='ID_1', right_index=True, how='left')
 .merge(df_input_clean, left_on='ID_2', right_index=True, how='left', suffixes=('_benchmark', '_input')))
output_full.to_excel("C:\\data\\HUKsvoc\\data\\20260403_output_large_FIX.xlsx", index=False)

In [ ]:
(output_full[
        output_full['ID_2'].isin(df_input[df_input[settings.INPUT_COLUMNS.ADDRESS].isna()][settings.INPUT_COLUMNS.ID].tolist())
]
 .to_excel("C:\\data\\HUKsvoc\\data\\20260403_output_large_MISSINGADDRESS_FIX.xlsx", index=False))

In [ ]:
(output_full[output_full['ID_2'].isin(list_idx_input)]
 .to_excel("C:\\data\\HUKsvoc\\data\\20260403_output_large_MAXMATCHES.xlsx", index=False))